## Setup

In [1]:
!pip install -q evaluate peft

import numpy as np
import pandas as pd
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
    pipeline,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
Device: cuda


## Dataset — Rotten Tomatoes



In [2]:
dataset = load_dataset('rotten_tomatoes')

# Quick description
print(dataset)
print('\nLabel distribution (train):')
print(pd.Series(dataset['train']['label']).value_counts().to_string())

# First sample
print('\nFirst training example:')
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

Label distribution (train):
1    4265
0    4265

First training example:
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


---

# Part 1 — RoBERTa: Zero-shot Inference



In [3]:
MODEL_ID = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

# Use top_k=None (modern equivalent of return_all_scores=True)
sentiment_task = pipeline(
    'sentiment-analysis',
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    top_k=None,  # returns all class scores instead of just the top one
    device=0 if device == 'cuda' else -1,
)

# Run inference on the full test split
test_data = list(dataset['test'])
test_texts = [s['text'] for s in test_data]
y_true = [s['label'] for s in test_data]

print(f'Running zero-shot inference on {len(test_texts)} reviews...')
results = sentiment_task(test_texts, batch_size=32)

# Show a few samples
for i in range(3):
    print(f'\n--- Review {i+1} ---')
    print(f'Text  : {test_texts[i][:120]}...')
    print(f'Scores: {results[i]}')

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Running zero-shot inference on 1066 reviews...

--- Review 1 ---
Text  : lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without ...
Scores: [{'label': 'positive', 'score': 0.9546050429344177}, {'label': 'neutral', 'score': 0.040233686566352844}, {'label': 'negative', 'score': 0.005161256529390812}]

--- Review 2 ---
Text  : consistently clever and suspenseful ....
Scores: [{'label': 'positive', 'score': 0.8883833885192871}, {'label': 'neutral', 'score': 0.1039101704955101}, {'label': 'negative', 'score': 0.007706433534622192}]

--- Review 3 ---
Text  : it's like a " big chill " reunion of the baader-meinhof gang , only these guys are more harmless pranksters than politic...
Scores: [{'label': 'negative', 'score': 0.7359194159507751}, {'label': 'neutral', 'score': 0.24242661893367767}, {'label': 'positive', 'score': 0.021653927862644196}]


In [4]:
# Convert pipeline outputs to binary predictions: 1 if best label is 'positive', else 0
y_pred = []
for pred in results:
    if isinstance(pred, dict):
        # Single dict (top-1 result)
        best_label = pred['label']
    else:
        # List of dicts (top_k=None or return_all_scores=True)
        best_label = max(pred, key=lambda x: x['score'])['label']
    y_pred.append(1 if best_label.lower() == 'positive' else 0)

print('Zero-shot RoBERTa — Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

acc_zeroshot_roberta = (np.array(y_pred) == np.array(y_true)).mean()

Zero-shot RoBERTa — Classification Report:
              precision    recall  f1-score   support

    Negative       0.68      0.94      0.79       533
    Positive       0.91      0.56      0.69       533

    accuracy                           0.75      1066
   macro avg       0.79      0.75      0.74      1066
weighted avg       0.79      0.75      0.74      1066



### Tokenizer & Embedding Inspection


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
encoder = AutoModel.from_pretrained(MODEL_ID)

prompt = 'I love this movie!'
inputs = tokenizer(prompt, return_tensors='pt')

with torch.no_grad():
    outputs = encoder(**inputs)

# Build a small DataFrame: token, ID, first 5 dims of its contextual embedding
token_embeddings = outputs.last_hidden_state[0]
input_ids = inputs['input_ids'][0]

inspect_df = pd.DataFrame([
    {
        'Token': tokenizer.decode(tid),
        'ID': tid.item(),
        'Embedding[:5]': emb[:5].numpy().round(3),
    }
    for tid, emb in zip(input_ids, token_embeddings)
])
inspect_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
classifier.out_proj.bias        | UNEXPECTED |  | 
classifier.dense.weight         | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
classifier.dense.bias           | UNEXPECTED |  | 
classifier.out_proj.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Token,ID,Embedding[:5]
0,<s>,0,"[-0.057, 0.674, -0.246, -0.402, 1.893]"
1,I,100,"[-0.087, 0.259, -0.221, 0.107, 1.524]"
2,love,657,"[-0.343, 0.489, 0.055, 0.023, 1.761]"
3,this,42,"[-0.181, 0.203, 0.141, -0.064, 1.545]"
4,movie,1569,"[-0.355, 0.292, 0.102, -0.031, 1.311]"
5,!,328,"[-0.118, 0.163, 0.205, 0.114, 1.614]"
6,</s>,2,"[-0.057, 0.675, -0.246, -0.402, 1.893]"


---

# Part 2 — RoBERTa: Linear Probing



In [6]:
# Reload tokenizer for the classification head setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Common metrics function for all RoBERTa experiments
metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [7]:
model_lp = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, ignore_mismatched_sizes=True
)

# Freeze the backbone, leave only the classifier trainable
for param in model_lp.roberta.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model_lp.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_lp.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs mod

Trainable: 592,130 / 124,647,170 (0.48%)


In [8]:
training_args = TrainingArguments(
    output_dir='roberta_linear_probing',
    eval_strategy='epoch',
    per_device_train_batch_size=32,
    num_train_epochs=5,
    report_to='none',
)

trainer_lp = Trainer(
    model=model_lp,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
trainer_lp.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.406668,0.819887
2,0.420004,0.384680,0.825516
3,0.420004,0.387967,0.824578
4,0.395357,0.386406,0.829268
5,0.395357,0.382510,0.823640


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1335, training_loss=0.40326700389161985, metrics={'train_runtime': 51.7377, 'train_samples_per_second': 824.35, 'train_steps_per_second': 25.803, 'total_flos': 1146363562650360.0, 'train_loss': 0.40326700389161985, 'epoch': 5.0})

In [9]:
predictions = trainer_lp.predict(tokenized_datasets['test'])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print('Linear Probing RoBERTa — Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

acc_linearprobing_roberta = (y_pred == y_true).mean()

Linear Probing RoBERTa — Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.85      0.83       533
    Positive       0.84      0.80      0.82       533

    accuracy                           0.82      1066
   macro avg       0.82      0.82      0.82      1066
weighted avg       0.82      0.82      0.82      1066



---

# Part 3 — RoBERTa: Fine-tuning


In [10]:
model_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, ignore_mismatched_sizes=True
)

training_args_ft = TrainingArguments(
    output_dir='roberta_full_finetuning',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=5,
    report_to='none',
)

trainer_full = Trainer(
    model=model_full,
    args=training_args_ft,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
trainer_full.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs mod

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.314504,0.863977
2,0.283392,0.394662,0.871482
3,0.283392,0.480020,0.872420
4,0.128736,0.563549,0.875235
5,0.128736,0.607355,0.872420


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1335, training_loss=0.17182694195808096, metrics={'train_runtime': 111.6541, 'train_samples_per_second': 381.983, 'train_steps_per_second': 11.957, 'total_flos': 1146363562650360.0, 'train_loss': 0.17182694195808096, 'epoch': 5.0})

In [11]:
pred_full = trainer_full.predict(tokenized_datasets['test'])
y_pred_full = np.argmax(pred_full.predictions, axis=-1)

print('Full Fine-tuning RoBERTa — Classification Report:')
print(classification_report(pred_full.label_ids, y_pred_full, target_names=['Negative', 'Positive']))

acc_fullft_roberta = (y_pred_full == pred_full.label_ids).mean()

Full Fine-tuning RoBERTa — Classification Report:
              precision    recall  f1-score   support

    Negative       0.86      0.89      0.88       533
    Positive       0.89      0.85      0.87       533

    accuracy                           0.87      1066
   macro avg       0.87      0.87      0.87      1066
weighted avg       0.87      0.87      0.87      1066



## Part 3b — Partial Fine-tuning (freeze first 6 layers)



In [12]:
model_partial = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, ignore_mismatched_sizes=True
)

# Start from all trainable, then freeze the first 6 encoder layers
for layer in model_partial.roberta.encoder.layer[:6]:
    for param in layer.parameters():
        param.requires_grad = False

trainable = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_partial.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs mod

Trainable: 82,119,938 / 124,647,170 (65.88%)


In [13]:
trainer_partial = Trainer(
    model=model_partial,
    args=training_args_ft,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
trainer_partial.train()

pred_partial = trainer_partial.predict(tokenized_datasets['test'])
y_pred_partial = np.argmax(pred_partial.predictions, axis=-1)

print('Partial Fine-tuning RoBERTa — Classification Report:')
print(classification_report(pred_partial.label_ids, y_pred_partial, target_names=['Negative', 'Positive']))

acc_partialft_roberta = (y_pred_partial == pred_partial.label_ids).mean()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.330317,0.852720
2,0.326751,0.330997,0.871482
3,0.326751,0.352732,0.865854
4,0.215017,0.383143,0.869606
5,0.215017,0.402514,0.868668


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Partial Fine-tuning RoBERTa — Classification Report:
              precision    recall  f1-score   support

    Negative       0.86      0.88      0.87       533
    Positive       0.88      0.86      0.87       533

    accuracy                           0.87      1066
   macro avg       0.87      0.87      0.87      1066
weighted avg       0.87      0.87      0.87      1066



---

# Part 4 — RoBERTa: LoRA (PEFT)


In [14]:
!pip install -q -U torchao
model_lora = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, ignore_mismatched_sizes=True
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['query', 'value'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.2 MB/s eta 0:00:00


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs mod

trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


In [15]:
trainer_lora = Trainer(
    model=model_lora,
    args=training_args_ft,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)
trainer_lora.train()

pred_lora = trainer_lora.predict(tokenized_datasets['test'])
y_pred_lora = np.argmax(pred_lora.predictions, axis=-1)

print('LoRA RoBERTa — Classification Report:')
print(classification_report(pred_lora.label_ids, y_pred_lora, target_names=['Negative', 'Positive']))

acc_lora_roberta = (y_pred_lora == pred_lora.label_ids).mean()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.378032,0.841463
2,0.392177,0.358019,0.847092
3,0.392177,0.355015,0.850844
4,0.340217,0.348986,0.850844
5,0.340217,0.349287,0.848968


LoRA RoBERTa — Classification Report:
              precision    recall  f1-score   support

    Negative       0.84      0.86      0.85       533
    Positive       0.86      0.84      0.85       533

    accuracy                           0.85      1066
   macro avg       0.85      0.85      0.85      1066
weighted avg       0.85      0.85      0.85      1066



---

# Part 5 — GPT-2 (Decoder Model)


In [16]:
GPT2_ID = 'gpt2'
generator = pipeline('text-generation', model=GPT2_ID, device=0 if device == 'cuda' else -1)

# Shuffle the test set so evaluation isn't biased by the dataset order
shuffled_test = dataset['test'].shuffle(seed=42)
gpt2_test = list(shuffled_test)
gpt2_texts = [s['text'] for s in gpt2_test]
gpt2_y_true = [s['label'] for s in gpt2_test]

gpt2_y_pred = []
for text in gpt2_texts:
    prompt = f'Review: {text[:300]}\nSentiment:'
    output = generator(
        prompt,
        max_new_tokens=1,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    prediction = output[0]['generated_text'].split('Sentiment:')[-1].strip().lower()
    gpt2_y_pred.append(1 if 'positive' in prediction else 0)

print('Zero-shot GPT-2 — Classification Report:')
print(classification_report(gpt2_y_true, gpt2_y_pred, target_names=['Negative', 'Positive']))

acc_zeroshot_gpt2 = (np.array(gpt2_y_pred) == np.array(gpt2_y_true)).mean()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_genera

Zero-shot GPT-2 — Classification Report:
              precision    recall  f1-score   support

    Negative       0.50      0.99      0.66       533
    Positive       0.25      0.00      0.01       533

    accuracy                           0.50      1066
   macro avg       0.37      0.50      0.33      1066
weighted avg       0.37      0.50      0.33      1066



## Part 5b — GPT-2 Linear Probing


In [17]:
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_ID)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

def tokenize_gpt2(examples):
    return gpt2_tokenizer(
        examples['text'], padding='max_length', truncation=True, max_length=128
    )

gpt2_tokenized = dataset.map(tokenize_gpt2, batched=True)

model_gpt2_lp = AutoModelForSequenceClassification.from_pretrained(GPT2_ID, num_labels=2)
model_gpt2_lp.config.pad_token_id = model_gpt2_lp.config.eos_token_id

# Freeze the transformer body
for param in model_gpt2_lp.transformer.parameters():
    param.requires_grad = False

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key                  | Status     | 
---------------------+------------+-
h.{0...11}.attn.bias | UNEXPECTED | 
score.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
training_args_gpt2 = TrainingArguments(
    output_dir='gpt2_linear_probing',
    eval_strategy='epoch',
    per_device_train_batch_size=32,
    num_train_epochs=5,
    report_to='none',
)

trainer_gpt2_lp = Trainer(
    model=model_gpt2_lp,
    args=training_args_gpt2,
    train_dataset=gpt2_tokenized['train'],
    eval_dataset=gpt2_tokenized['test'],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=gpt2_tokenizer),
)
trainer_gpt2_lp.train()

pred = trainer_gpt2_lp.predict(gpt2_tokenized['test'])
y_pred = np.argmax(pred.predictions, axis=-1)

print('Linear Probing GPT-2 — Classification Report:')
print(classification_report(pred.label_ids, y_pred, target_names=['Negative', 'Positive']))

acc_linearprobing_gpt2 = (y_pred == pred.label_ids).mean()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.667089,0.661351
2,0.966286,0.632445,0.689493
3,0.966286,0.617844,0.673546
4,0.661150,0.605419,0.693246
5,0.661150,0.601772,0.700750


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Linear Probing GPT-2 — Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.60      0.67       533
    Positive       0.67      0.80      0.73       533

    accuracy                           0.70      1066
   macro avg       0.71      0.70      0.70      1066
weighted avg       0.71      0.70      0.70      1066



## Part 5c — GPT-2 Full Fine-tuning

In [19]:
model_gpt2_full = AutoModelForSequenceClassification.from_pretrained(
    GPT2_ID, num_labels=2, ignore_mismatched_sizes=True
)
model_gpt2_full.config.pad_token_id = gpt2_tokenizer.pad_token_id

training_args_gpt2_ft = TrainingArguments(
    output_dir='gpt2_full_finetuning',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=5,
    report_to='none',
)

trainer_gpt2_full = Trainer(
    model=model_gpt2_full,
    args=training_args_gpt2_ft,
    train_dataset=gpt2_tokenized['train'],
    eval_dataset=gpt2_tokenized['test'],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=gpt2_tokenizer),
)
trainer_gpt2_full.train()

pred = trainer_gpt2_full.predict(gpt2_tokenized['test'])
y_pred = np.argmax(pred.predictions, axis=-1)

print('Full Fine-tuning GPT-2 — Classification Report:')
print(classification_report(pred.label_ids, y_pred, target_names=['Negative', 'Positive']))

acc_fullft_gpt2 = (y_pred == pred.label_ids).mean()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key                  | Status     | 
---------------------+------------+-
h.{0...11}.attn.bias | UNEXPECTED | 
score.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.381822,0.822702
2,0.446901,0.331495,0.859287
3,0.446901,0.327404,0.868668
4,0.251684,0.336287,0.863977
5,0.251684,0.356270,0.864916


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Full Fine-tuning GPT-2 — Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.85      0.86       533
    Positive       0.85      0.88      0.87       533

    accuracy                           0.86      1066
   macro avg       0.87      0.86      0.86      1066
weighted avg       0.87      0.86      0.86      1066



---

# Final Summary



In [20]:
summary = pd.DataFrame([
    ('RoBERTa', 'Zero-shot',                       '0%',     acc_zeroshot_roberta),
    ('RoBERTa', 'Linear Probing',                  '~0.0%',  acc_linearprobing_roberta),
    ('RoBERTa', 'Full Fine-tuning',                '100%',   acc_fullft_roberta),
    ('RoBERTa', 'Partial FT (6 layers frozen)',    '~50%',   acc_partialft_roberta),
    ('RoBERTa', 'LoRA (r=8)',                      '0.71%',  acc_lora_roberta),
    ('GPT-2',   'Zero-shot prompting',             '0%',     acc_zeroshot_gpt2),
    ('GPT-2',   'Linear Probing',                  '~0.0%',  acc_linearprobing_gpt2),
    ('GPT-2',   'Full Fine-tuning',                '100%',   acc_fullft_gpt2),
], columns=['Model', 'Strategy', 'Trainable Params', 'Accuracy'])

summary['Accuracy'] = summary['Accuracy'].apply(lambda x: f'{100*x:.2f}%')
print(summary.to_string(index=False))

  Model                     Strategy Trainable Params Accuracy
RoBERTa                    Zero-shot               0%   75.14%
RoBERTa               Linear Probing            ~0.0%   82.36%
RoBERTa             Full Fine-tuning             100%   87.24%
RoBERTa Partial FT (6 layers frozen)             ~50%   86.87%
RoBERTa                   LoRA (r=8)            0.71%   84.90%
  GPT-2          Zero-shot prompting               0%   49.62%
  GPT-2               Linear Probing            ~0.0%   70.08%
  GPT-2             Full Fine-tuning             100%   86.49%
